# [LAB] XAI (Explainable AI) with Wine Dataset

## Overview

In this lab you will apply **Explainable AI (XAI)** techniques to classify wine:  
**Can a machine tell Red Wine from White Wine using only three numbers?**

| Feature | Description |
|---------|-------------|
| `alcohol` | Alcohol content (%) |
| `sugar` | Residual sugar (g/L) |
| `pH` | Acidity level |
| `class` | **Target**: 0 = Red Wine, 1 = White Wine |

### Tasks
Fill in every cell marked with `# TODO` and answer the questions in the markdown cells.

### XAI Techniques to Practice
1. **Feature Importance** — Which feature does the model rely on most?
2. **Partial Dependence Plots (PDP)** — How does one feature change the prediction?
3. **2D PDP** — How do two features interact?
4. **Confusion Matrix** — Where does the model go wrong?
5. **Hyperparameter Tuning (GridSearchCV)** — How do we improve the model?

> 💡 Stuck? Refer to the **solution notebook** `[0430]XAI_wine_solution.ipynb`.

## Section 0 — Install Dependencies

Run this cell once to install all required libraries.

In [ ]:
# TODO: Run this cell as-is. It installs all required packages.
%pip install xgboost scikit-learn matplotlib pandas numpy --quiet

## Section 1 — Load the Wine Dataset

The data file `wine.csv` is in the **same folder** as this notebook.  
We use `pd.read_csv()` to read it into a pandas DataFrame.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -- Find wine.csv in the current or parent folder
for candidate in [Path('./wine.csv'), Path('../wine.csv')]:
    if candidate.exists():
        csv_path = candidate.resolve()
        break
else:
    raise FileNotFoundError("wine.csv not found. Make sure wine.csv is in the same folder.")

print(f"Loading data from: {csv_path}")

# TODO (1): Load the CSV file into a variable called 'wine'
# Hint: pd.read_csv(csv_path)
wine = # YOUR CODE HERE

# TODO (2): Display the first 5 rows of the DataFrame
# Hint: wine.head()
# YOUR CODE HERE

In [ ]:
# TODO (3): Print the shape of the dataset (rows, columns)
print("Dataset shape:", # YOUR CODE HERE
)

# TODO (4): Call wine.info() to see column types and missing values
# YOUR CODE HERE

# TODO (5): How many red wines (class=0) and white wines (class=1) are there?
# Hint: wine['class'].value_counts()
print("\nClass counts:")
# YOUR CODE HERE

In [ ]:
# TODO (6): Plot histograms of each feature, coloured by wine type
# This helps you see which features naturally separate red from white wine

feature_names = ['alcohol', 'sugar', 'pH']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Feature Distributions by Wine Type', fontsize=14)

for ax, feature in zip(axes, feature_names):
    for cls, color, label in [(0, '#c0392b', 'Red Wine'), (1, '#f5cba7', 'White Wine')]:
        subset = wine[wine['class'] == cls][feature]
        # TODO: Plot a histogram of 'subset' on 'ax' with 40 bins
        # Hint: ax.hist(subset, bins=40, alpha=0.6, color=color, label=label)
        # YOUR CODE HERE
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.set_title(f'{feature}')
    ax.legend()

plt.tight_layout()
plt.show()

**Q1:** Looking at the histograms, which feature seems to best separate red wine from white wine? Why?

**Answer:** *(Write your answer here)*

## Section 2 — Train / Test Split

We split the data into:
- **Training set (80%)**: Used to teach the model  
- **Test set (20%)**: Used to check if the model works on new, unseen data

We also use `stratify=y` so that both splits have the same ratio of red/white wine.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = ['alcohol', 'sugar', 'pH']
TARGET_COL   = 'class'

# Convert to numpy arrays (XGBoost works with numpy)
X = wine[FEATURE_COLS].to_numpy()           # input features
y = wine[TARGET_COL].astype(int).to_numpy() # labels: 0=Red, 1=White

# TODO (7): Split X and y into X_train, X_test, y_train, y_test
# Use test_size=0.2, random_state=42, stratify=y
# Hint: train_test_split(X, y, test_size=..., random_state=..., stratify=...)
X_train, X_test, y_train, y_test = # YOUR CODE HERE

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## Section 2.5 — Information Gain with Gini and Cross-Entropy (Formula Drill)

In this section, you will practice the exact impurity calculations used by decision trees.

### Mini dataset (binary classification)
We classify `Exercise = Yes/No` with 6 samples:

| ID | Weather | Temperature | Exercise |
|---:|---------|-------------|----------|
| 1 | Sunny  | Hot  | No  |
| 2 | Sunny  | Mild | Yes |
| 3 | Cloudy | Mild | Yes |
| 4 | Rain   | Mild | Yes |
| 5 | Rain   | Cold | No  |
| 6 | Cloudy | Cold | Yes |

Candidate split A: `Weather == Cloudy?`

Candidate split B: `Temperature == Cold?`

### Core formulas
For class probabilities $p_k$:

$$
\mathrm{Gini}(S)=1-\sum_k p_k^2
$$

$$
\mathrm{Entropy}(S)= -\sum_k p_k\log_2 p_k
$$

For a split into children $S_L, S_R$:

$$
\mathrm{WeightedImpurity}(S)=\frac{|S_L|}{|S|}\,\mathrm{Impurity}(S_L)+\frac{|S_R|}{|S|}\,\mathrm{Impurity}(S_R)
$$

$$
\mathrm{Gain}=\mathrm{Impurity}(S)-\mathrm{WeightedImpurity}(S)
$$

Cross-entropy notation:

$$
H(p,q)=-\sum_k p_k\log_2 q_k
$$

If $q=p$, then $H(p,p)=H(p)$, so entropy is a special case of cross-entropy.

### Target check values (use to verify your work)
- Parent: Yes=4, No=2
- Parent Gini: $1-(4/6)^2-(2/6)^2=4/9\approx0.444$
- Parent Entropy: $-(4/6)\log_2(4/6)-(2/6)\log_2(2/6)\approx0.918$

Split A (`Cloudy?`):
- Weighted Gini $=0.333$, Gini Gain $=0.111$
- Weighted Entropy $=0.667$, Information Gain $=0.251$

Split B (`Cold?`):
- Weighted Gini $=0.417$, Gini Gain $=0.027$
- Weighted Entropy $\approx0.874$, Information Gain $\approx0.044$

Conclusion: split A is better (larger gain under both criteria).

In [ ]:
import math
import pandas as pd

# TODO (2.5-1): Build the mini dataset from the table above
mini = pd.DataFrame({
    'weather': ['Sunny', 'Sunny', 'Cloudy', 'Rain', 'Rain', 'Cloudy'],
    'temperature': ['Hot', 'Mild', 'Mild', 'Mild', 'Cold', 'Cold'],
    'exercise': ['No', 'Yes', 'Yes', 'Yes', 'No', 'Yes'],
})


def gini_from_counts(counts):
    total = sum(counts)
    probs = [c / total for c in counts if c > 0]
    return 1 - sum(p**2 for p in probs)


def entropy_from_counts(counts):
    total = sum(counts)
    probs = [c / total for c in counts if c > 0]
    return -sum(p * math.log2(p) for p in probs)


def split_scores(df, col, positive_value):
    left = df[df[col] == positive_value]['exercise']
    right = df[df[col] != positive_value]['exercise']

    parent_counts = df['exercise'].value_counts().reindex(['No', 'Yes'], fill_value=0).tolist()
    left_counts = left.value_counts().reindex(['No', 'Yes'], fill_value=0).tolist()
    right_counts = right.value_counts().reindex(['No', 'Yes'], fill_value=0).tolist()

    parent_gini = gini_from_counts(parent_counts)
    parent_entropy = entropy_from_counts(parent_counts)

    w_left = len(left) / len(df)
    w_right = len(right) / len(df)

    weighted_gini = w_left * gini_from_counts(left_counts) + w_right * gini_from_counts(right_counts)
    weighted_entropy = w_left * entropy_from_counts(left_counts) + w_right * entropy_from_counts(right_counts)

    return {
        'parent_gini': parent_gini,
        'parent_entropy': parent_entropy,
        'weighted_gini': weighted_gini,
        'gini_gain': parent_gini - weighted_gini,
        'weighted_entropy': weighted_entropy,
        'info_gain': parent_entropy - weighted_entropy,
    }


# TODO (2.5-2): Compute scores for split A and split B
score_a = split_scores(mini, 'weather', 'Cloudy')
score_b = split_scores(mini, 'temperature', 'Cold')

print('Parent Gini       :', round(score_a['parent_gini'], 3))
print('Parent Entropy    :', round(score_a['parent_entropy'], 3))

print('\nSplit A (Cloudy?)')
print('  Weighted Gini   :', round(score_a['weighted_gini'], 3))
print('  Gini Gain       :', round(score_a['gini_gain'], 3))
print('  Weighted Entropy:', round(score_a['weighted_entropy'], 3))
print('  Info Gain       :', round(score_a['info_gain'], 3))

print('\nSplit B (Cold?)')
print('  Weighted Gini   :', round(score_b['weighted_gini'], 3))
print('  Gini Gain       :', round(score_b['gini_gain'], 3))
print('  Weighted Entropy:', round(score_b['weighted_entropy'], 3))
print('  Info Gain       :', round(score_b['info_gain'], 3))

best = 'A (Cloudy?)' if score_a['info_gain'] > score_b['info_gain'] else 'B (Cold?)'
print('\nBest split by information gain:', best)

## Section 3 — Train the XGBoost Model

**XGBoost** builds decision trees one at a time, where each new tree corrects the mistakes of the previous ones.  
This is called **boosting**.

```
Tree 1 → makes some mistakes
Tree 2 → focuses on Tree 1's mistakes
Tree 3 → focuses on Tree 2's mistakes
...repeat...
Final prediction = all trees vote together
```

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# TODO (8): Create an XGBClassifier with eval_metric='logloss' and random_state=42
# Hint: XGBClassifier(eval_metric='logloss', random_state=42)
model = # YOUR CODE HERE

# TODO (9): Train the model on the training data
# Hint: model.fit(X_train, y_train)
# YOUR CODE HERE

# TODO (10): Make predictions on the test set
# Hint: model.predict(X_test)
y_pred_test = # YOUR CODE HERE

# TODO (11): Calculate and print the test accuracy
# Hint: accuracy_score(y_test, y_pred_test)
test_acc = # YOUR CODE HERE
print(f"Test Accuracy: {test_acc:.2%}")

**Q2:** What does 'accuracy' measure? Is 90% accuracy always good? When might it be misleading?

**Answer:** *(Write your answer here)*

## Section 4 — XAI Technique 1: Confusion Matrix

A confusion matrix shows what the model got **right** and what it got **wrong**.

```
               Predicted Red   Predicted White
Actual Red    [ True Negative   False Positive ]
Actual White  [ False Negative  True Positive  ]
```

- **True Positive (TP)**: Correctly predicted White Wine
- **True Negative (TN)**: Correctly predicted Red Wine  
- **False Positive (FP)**: Said "White" but it was Red → **mislabeled red as white**
- **False Negative (FN)**: Said "Red" but it was White → **mislabeled white as red**

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report

# TODO (12): Compute the confusion matrix
# Hint: confusion_matrix(y_test, y_pred_test)
cm = # YOUR CODE HERE
print("Confusion Matrix:")
print(cm)

In [ ]:
# -- Visualise the confusion matrix (code provided — study it!)
def plot_confusion_matrix(cm, class_names, title='Confusion Matrix'):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im)
    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, fontsize=12)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names, fontsize=12)
    total = cm.sum()
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, f"{cm[i,j]}\n({cm[i,j]/total:.1%})",
                ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black', fontsize=13)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(cm, ['Red (0)', 'White (1)'])

# TODO (13): Print the classification report for detailed metrics
# Hint: print(classification_report(y_test, y_pred_test, target_names=['Red Wine', 'White Wine']))
# YOUR CODE HERE

**Q3:** Look at the confusion matrix.
- How many red wines were incorrectly predicted as white? (False Positives)
- How many white wines were incorrectly predicted as red? (False Negatives)
- Which type of error does the model make more often?

**Answer:** *(Write your answer here)*

## Section 5 — XAI Technique 2: Feature Importance

Feature importance answers: **"Which feature does the model rely on most?"**  
XGBoost assigns a score to each feature based on how useful it is for making predictions.

In [ ]:
from xgboost import plot_importance

# TODO (14): Extract the feature importance scores from the trained model
# Hint: model.feature_importances_  (this is an array of 3 numbers)
importances = # YOUR CODE HERE

print("Feature Importances:")
for name, score in zip(FEATURE_COLS, importances):
    print(f"  {name:10s}: {score:.4f}")

In [ ]:
# TODO (15): Create a horizontal bar chart showing feature importances
# Steps:
#   1. Sort the features by importance (np.argsort)
#   2. Plot using plt.barh()
#   3. Add labels and a title

# YOUR CODE HERE
# (Reference the solution notebook if you need help)

# Tip: Also try the built-in XGBoost function:
# from xgboost import plot_importance
# plot_importance(model)
# plt.show()

**Q4:** Which feature has the highest importance score? Does this match your visual observation from the histograms in Section 1?

**Answer:** *(Write your answer here)*

## Section 6 — XAI Technique 3: 1D Partial Dependence Plot (PDP)

A **Partial Dependence Plot** shows:
> *"If I change only the alcohol content (while keeping sugar and pH fixed), how does P(White Wine) change?"*

- **Y-axis**: Predicted probability of White Wine (0 = Red, 1 = White)
- **X-axis**: The feature value
- The **red dashed line** at 0.5 is the decision boundary

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# TODO (16): Create 1D Partial Dependence Plots for all three features
# Create a figure with 1 row and 3 subplots (one per feature)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('1D Partial Dependence Plots — Effect of Each Feature on P(White Wine)',
             fontsize=13, fontweight='bold')

for ax, (idx, feature) in zip(axes, enumerate(FEATURE_COLS)):
    # TODO: Call PartialDependenceDisplay.from_estimator() for each feature
    # Parameters:
    #   estimator = model
    #   X = X_train
    #   features = [idx]   (index of the feature)
    #   feature_names = FEATURE_COLS
    #   ax = ax
    # YOUR CODE HERE

    ax.set_title(f'Effect of {feature}', fontsize=11)
    ax.set_ylabel('P(White Wine)')
    # Add a red dashed line at y=0.5 (the decision boundary)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.6, label='Decision boundary')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Q5:** Describe the shape of the PDP for `sugar`:
- As sugar increases from 1 g/L to 20 g/L, does P(White Wine) increase, decrease, or stay flat?
- At approximately what sugar value does the model cross the 0.5 decision boundary?
- Can you explain this chemically? (Hint: What kind of wines have high residual sugar?)

**Answer:** *(Write your answer here)*

## Section 7 — XAI Technique 4: 2D Partial Dependence Plot

A **2D PDP** shows how **two features together** affect the prediction.  
The colour shows P(White Wine): lighter = more likely Red, darker = more likely White.

In [ ]:
# TODO (17): Create two 2D Partial Dependence Plots
#   Plot 1: alcohol (index 0) vs sugar (index 1)
#   Plot 2: alcohol (index 0) vs pH (index 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('2D Partial Dependence Plots (Feature Interactions)', fontsize=13)

# Plot 1: alcohol vs sugar
# TODO: Use PartialDependenceDisplay.from_estimator with features=[(0, 1)]
# YOUR CODE HERE
axes[0].set_title('Alcohol × Sugar', fontsize=12)

# Plot 2: alcohol vs pH
# TODO: Use PartialDependenceDisplay.from_estimator with features=[(0, 2)]
# YOUR CODE HERE
axes[1].set_title('Alcohol × pH', fontsize=12)

plt.tight_layout()
plt.show()

**Q6:** Look at the Alcohol × Sugar 2D PDP:
- Which corner (top-right / bottom-left / top-left / bottom-right) is most strongly predicted as White Wine?
- Does this make intuitive sense?

**Answer:** *(Write your answer here)*

## Section 8 — Visualise a Decision Tree

XGBoost is made of many trees, but we can look at one tree to understand its structure.  
We train a *simpler* model (`max_depth=2`) so the tree is easy to read.

In [ ]:
from xgboost import plot_tree

# TODO (18): Train a simple XGBoost model with n_estimators=5, max_depth=2
# This makes the trees small enough to visualise
simple_model = XGBClassifier(
    n_estimators=5,
    max_depth=2,
    learning_rate=0.3,
    eval_metric='logloss',
    random_state=42
)
# TODO: Fit simple_model on the training data
# YOUR CODE HERE

# -- Plot the first tree in the ensemble
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    simple_model,
    num_trees=0,   # Show the first tree (index 0)
    ax=ax,
    rankdir='LR'   # Draw left-to-right
)
ax.set_title('First Decision Tree in the XGBoost Ensemble (max_depth=2)', fontsize=13)
plt.tight_layout()
plt.show()

print("Hint: 'f0'=alcohol, 'f1'=sugar, 'f2'=pH")
print("Positive leaf score → leans toward White Wine")
print("Negative leaf score → leans toward Red Wine")

**Q7:** Look at the decision tree:
- What is the very first question (root node) the tree asks?
- Which feature does this tree find most important at the top level?

**Answer:** *(Write your answer here)*

## Section 9 — Predict for a Specific Wine

Now let's use the model to classify a wine with specific chemical properties.

In [ ]:
# -- A hypothetical wine sample
# Feel free to change these values and observe how the prediction changes!
custom_wine = {
    'alcohol': 12.5,  # Moderate alcohol
    'sugar':   8.0,   # Medium-high sugar (could be semi-sweet white)
    'pH':      3.1,   # Fairly acidic
}
sample = np.array([list(custom_wine.values())])

# TODO (19): Use model.predict_proba() to get the probability of each class
# predict_proba() returns [[P(Red), P(White)]]
proba = # YOUR CODE HERE

# TODO (20): Use model.predict() to get the final class label (0 or 1)
label = # YOUR CODE HERE

print("Custom wine sample:")
for k, v in custom_wine.items():
    print(f"  {k}: {v}")
print(f"\nP(Red Wine)  : {proba[0][0]:.2%}")
print(f"P(White Wine): {proba[0][1]:.2%}")
print(f"Prediction   : {'White Wine 🤍' if label[0] == 1 else 'Red Wine 🍷'}")

**Q8:** Try changing `sugar` to `1.0` (a very dry wine). Does the prediction change?  
Then try `alcohol=8.5, sugar=25.0, pH=3.5`. What does the model predict?

**Answer:** *(Write your answer here)*

## Section 10 — Hyperparameter Tuning with GridSearchCV

**Hyperparameters** are settings we choose *before* training. The model cannot learn them automatically.  

Key XGBoost hyperparameters:
| Parameter | Meaning |
|-----------|--------|
| `max_depth` | How deep each tree can grow (deeper = more complex) |
| `learning_rate` | How much each tree corrects previous errors |
| `n_estimators` | How many trees to build |

**GridSearchCV** automatically tries all combinations and picks the best one using cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# TODO (21): Define a parameter grid to search over
# Try: max_depth = [3, 5, 7]
#      learning_rate = [0.05, 0.1]
#      n_estimators = [50, 100, 200]
param_grid = {
    'max_depth':     # YOUR CODE HERE,
    'learning_rate': # YOUR CODE HERE,
    'n_estimators':  # YOUR CODE HERE,
}

# TODO (22): Create a GridSearchCV object with:
#   estimator = XGBClassifier(eval_metric='logloss', random_state=42)
#   param_grid = param_grid
#   scoring = 'accuracy'
#   cv = 5
#   n_jobs = -1
print("Running GridSearchCV... (this may take ~30 seconds)")
gs = GridSearchCV(
    # YOUR CODE HERE
)

# TODO (23): Fit the GridSearchCV on training data
# YOUR CODE HERE

# -- Show best results
print(f"\nBest hyperparameters: {gs.best_params_}")
print(f"Best CV accuracy:     {gs.best_score_:.4f}")

In [ ]:
# TODO (24): Evaluate the best model on the test set
# Hint: gs.best_estimator_ gives you the best model found by GridSearchCV
best_model = gs.best_estimator_

# TODO: Compute and print the test accuracy of best_model
# YOUR CODE HERE

print(f"Default model test accuracy: {test_acc:.2%}")
# TODO: Print the tuned model test accuracy and compare
# YOUR CODE HERE

**Q9:** Did GridSearchCV improve the accuracy? By how much?  
What were the best hyperparameters found?

**Answer:** *(Write your answer here)*

## Section 11 — Final Reflection

Answer these questions in your own words:

**Q10 (Bonus):** In a previous lab (`[0410]`), we used Random Forest and other ensemble methods on the same wine dataset.  
Random Forest got ~89% test accuracy. How does XGBoost compare?

XGBoost uses **sequential boosting** (each tree fixes previous mistakes), while Random Forest uses **parallel bagging** (trees are independent).  
In your opinion, when would you prefer one over the other?

**Answer:** *(Write your answer here)*

**Q11:** Why is XAI (Explainable AI) important for real-world applications?  
Give one example where knowing *why* a model made a decision is just as important as the accuracy itself.

**Answer:** *(Write your answer here)*

In [ ]:
# TODO (25): EXTENSION — Reproduce the feature importance bar chart for the tuned (best) model
# Compare it with the default model from Section 5.
# Did the feature importance ranking change after hyperparameter tuning?

# YOUR CODE HERE

---

## Checklist — Before Submitting

Make sure you have completed:

- [ ] Section 1: Loaded dataset, printed info, plotted histograms
- [ ] Section 2: Train/test split done correctly
- [ ] Section 3: XGBoost model trained and accuracy reported
- [ ] Section 4: Confusion matrix computed and visualised
- [ ] Section 5: Feature importance extracted and plotted
- [ ] Section 6: 1D PDPs plotted for all three features
- [ ] Section 7: 2D PDPs plotted for two feature pairs
- [ ] Section 8: Decision tree visualised
- [ ] Section 9: Single prediction made with custom wine sample
- [ ] Section 10: GridSearchCV run and best model evaluated
- [ ] All **Q1–Q9** answered in markdown cells

**Optional (Q10, Q11, Q25):** Complete these for bonus marks!